# SCUC - In-Class Example 4 (Method 2: indexed sets)

**Original AMPL author:** Xingpeng Li - Associate Professor, Dept. of Electrical and Computer Engineering, University of Houston (UH), Houston, TX, USA (Senior Member, IEEE). Email: xli83@central.uh.edu

**Converted by:** Haoxiang Wan - PhD student of Dr. Xingpeng Li (AMPL -> Pyomo + Gurobi)

Same problem as e4 but written with `GEN` / `PERIOD` index sets, mirroring the AMPL set/param formulation.

In [1]:
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, Objective, Constraint, SolverFactory,
    Binary, minimize, value
)

# ---- Data: loaded from external file 'SCUC_IC_e4_Method2_data.txt' ----
import sys, pathlib
# Make the ampl_data parser importable (one level up from this notebook)
_pkg = pathlib.Path.cwd().parent
if str(_pkg) not in sys.path: sys.path.insert(0, str(_pkg))
from ampl_data import parse_ampl_data

_d = parse_ampl_data('SCUC_IC_e4_Method2_data.txt')

GEN_data     = _d['GEN']
PERIOD_data  = _d['PERIOD']
gen_min      = _d['gen_min']
gen_max      = _d['gen_max']
gen_RRlimit  = _d['gen_RRlimit']
gen_OpCost   = _d['gen_OpCost']
gen_NlCost   = _d['gen_NlCost']
gen_SuCost   = _d['gen_SuCost']
Time_TotalPd = _d['Time_TotalPd']

BigM = 1e3
m = ConcreteModel()
m.GEN    = Set(initialize=GEN_data, ordered=True)
m.PERIOD = Set(initialize=PERIOD_data, ordered=True)

m.gen_min     = Param(m.GEN, initialize=gen_min)
m.gen_max     = Param(m.GEN, initialize=gen_max)
m.gen_RRlimit = Param(m.GEN, initialize=gen_RRlimit)
m.gen_OpCost  = Param(m.GEN, initialize=gen_OpCost)
m.gen_NlCost  = Param(m.GEN, initialize=gen_NlCost)
m.gen_SuCost  = Param(m.GEN, initialize=gen_SuCost)
m.Time_TotalPd = Param(m.PERIOD, initialize=Time_TotalPd)

m.u  = Var(m.GEN, m.PERIOD, domain=Binary)
m.v  = Var(m.GEN, m.PERIOD, domain=Binary)
m.Pg = Var(m.GEN, m.PERIOD)

m.obj = Objective(
    rule=lambda mm: sum(mm.gen_OpCost[g]*mm.Pg[g,t]
                       + mm.gen_NlCost[g]*mm.u[g,t]
                       + mm.gen_SuCost[g]*mm.v[g,t]
                       for g in mm.GEN for t in mm.PERIOD),
    sense=minimize
)

m.PowerBalance   = Constraint(m.PERIOD,
    rule=lambda mm, t: sum(mm.Pg[g, t] for g in mm.GEN) == mm.Time_TotalPd[t])
m.genLimit_Min   = Constraint(m.GEN, m.PERIOD,
    rule=lambda mm, g, t: mm.gen_min[g]*mm.u[g,t] <= mm.Pg[g,t])
m.genLimit_Max   = Constraint(m.GEN, m.PERIOD,
    rule=lambda mm, g, t: mm.Pg[g,t] <= mm.gen_max[g]*mm.u[g,t])

def rr_up(mm, g, t):
    if t == mm.PERIOD.first():
        return Constraint.Skip
    tp = mm.PERIOD.prev(t)
    return mm.Pg[g,t] - mm.Pg[g,tp] <= mm.gen_RRlimit[g]*mm.u[g,tp] + BigM*mm.v[g,t]

def rr_dn(mm, g, t):
    if t == mm.PERIOD.first():
        return Constraint.Skip
    tp = mm.PERIOD.prev(t)
    return mm.Pg[g,tp] - mm.Pg[g,t] <= mm.gen_RRlimit[g]*mm.u[g,t] + BigM*(mm.v[g,t] - mm.u[g,t] + mm.u[g,tp])

m.genRRLimit_Up = Constraint(m.GEN, m.PERIOD, rule=rr_up)
m.genRRLimit_Dn = Constraint(m.GEN, m.PERIOD, rule=rr_dn)

def vu_rule(mm, g, t):
    if t == mm.PERIOD.first():
        return mm.v[g,t] >= mm.u[g,t]
    return mm.v[g,t] >= mm.u[g,t] - mm.u[g, mm.PERIOD.prev(t)]
m.genVU = Constraint(m.GEN, m.PERIOD, rule=vu_rule)

model = m

In [2]:
# ---- Solve with Gurobi ----
solver = SolverFactory('gurobi')
solver.options['MIPGap'] = 0.0
solver.options['TimeLimit'] = 90
results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)
m = model
print("g  t   v   u    Pg")
for g in m.GEN:
    for t in m.PERIOD:
        print(f"{g}  {t}   {int(round(value(m.v[g,t])))}   {int(round(value(m.u[g,t])))}   {value(m.Pg[g,t]):.3f}")

Read LP format model from file C:\Users\hwan6\AppData\Local\Temp\tmpppogwxp3.pyomo.lp


Reading time = 0.00 seconds
x1: 18 rows, 12 columns, 48 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
TimeLimit  90
MIPGap  0



Optimize a model with 18 rows, 12 columns and 48 nonzeros
Model fingerprint: 0x6b770714
Variable types: 4 continuous, 8 integer (8 binary)
Coefficient statistics:


  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+01, 8e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [7e+01, 1e+02]


Presolve removed 8 rows and 6 columns


...

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time



*    0     0               0    3550.0000000 3550.00000  0.00%     -    0s


Explored 1 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 20 (of 20 available processors)

Solution count 5: 3550 4100 4200 ... 4900

Optimal solution found (tolerance 0.00e+00)


Best objective 3.550000000000e+03, best bound 3.550000000000e+03, gap 0.0000%


ok optimal
g  t   v   u    Pg
1  1   1   1   70.000
1  2   0   1   80.000
2  1   0   0   0.000
2  2   1   1   30.000
